# memopro 대 Unsloth: 같은 16비트 LoRA (docs/research/0227, E046)

같은 T4에서 같은 작업(WikiText-2, 512토큰, 배치 1, 10단계, LoRA r 8 q/k/v/o, AdamW 2e-4)을 세 방식으로 잰다: Hugging Face PEFT(그냥), memopro(가중치 흘려 쓰기), Unsloth(별도 가상환경). 모델은 Qwen2.5-3B와 7B. 7B는 16비트로 T4에 들어가지 않는다.

| 확인 | 기준(0227) |
|---|---|
| U1 (관문) | 7B 16비트: Unsloth는 완주 못 하고 memopro는 두 예산 모두 완주 |
| U2 (관문) | memopro 손실이 예산과 무관하게 비트 동일 |
| U3·U4·U5 | 3B 속도·메모리, 7B Unsloth 4비트 대 memopro 16비트, 손실 차이(보고) |

- 예상 시간: 45~60분(Unsloth 설치 5~10분 포함)
- 준비: 런타임 **T4 GPU**, Drive `memopro_colab/install/`에 소스 묶음 `memopro-src.tar.gz`(이미 있으면 그대로, 빌드는 Drive에 보관된 것을 다시 쓴다)

**준비 (처음 한 번, 세 노트북 공통)**
1. 런타임 → 런타임 유형 변경 → **T4 GPU**.
2. Google Drive에 `memopro_colab/install/` 폴더를 만들고 memopro 소스 배포본 `memopro-0.1.0a1.tar.gz`를 올린다(Linux wheel이 있으면 그것을 쓴다). 소스 배포본은 첫 실행 때 Rust로 빌드한다(3~5분).
   - 또는 Colab 왼쪽 열쇠(보안 비밀)에 `GITHUB_TOKEN`을 **직접** 등록하면 저장소에서 설치한다.
3. 셀을 실행한다. Drive 연결 허용 창이 뜨면 허용한다. 설정은 셀 맨 위(`SETTINGS`)에서 바꾼다.

**결과**: `MyDrive/memopro_colab/results/unsloth/<실행 id>/`
- `summary.md`·`summary.csv`: 비교 표와 읽는 법
- `cases/*.json`: 경우마다 모든 측정값(메모리, 시간, memopro 보고, 오류와 추적)
- `logs/*.log`: 경우마다 프로세스 출력 전체
- `timelines/*.csv`: 0.5초 간격 GPU·호스트 메모리와 GPU 사용률
- `env.json`: GPU, 판, memopro 빌드 확인
- 같은 이름의 `.zip`: 폴더 전체(공유용)

**끊겼을 때**: 같은 셀을 다시 실행하면 끝난 경우는 건너뛰고 이어서 한다(`NEW_RUN = True`면 새로 시작). 모델은 Drive의 `hf_cache/`에 남아 다시 받지 않는다. 세 노트북은 같은 Drive 폴더와 모델 캐시를 함께 쓴다.

**공정성**: 모든 경우는 **새 프로세스**에서 돈다(앞 경우의 메모리가 섞이지 않게, 0055). 여러 모델을 **동시에** 한 GPU에서 재면 서로의 측정을 오염시키므로(0085), 여러 모델은 목록으로 차례로 잰다.


In [ ]:
# ==== SETTINGS (edit here) ===========================================================
# E046 (pre-registered in docs/research/0227): the same 16-bit LoRA with Hugging Face PEFT,
# memopro (weights streamed) and Unsloth (in its own virtual environment) on a T4.
# About 45-60 minutes (Unsloth install ~5-10 min, 7B download ~15 GB if not cached).
MODELS = {  # model -> two host budgets for memopro (bytes)
    "Qwen/Qwen2.5-3B-Instruct": [2 << 30, 3 << 30],
    "Qwen/Qwen2.5-7B-Instruct": [4 << 30, 6 << 30],
}
LORA = {"steps": 10, "seq": 512}
UNSLOTH_4BIT = ["Qwen/Qwen2.5-7B-Instruct"]  # QLoRA reference (lossy), 7B only
TIMEOUT_S = 40 * 60
# ==== (settings shared by all cells) ================================================
DRIVE_ROOT = "/content/drive/MyDrive/memopro_colab"
GIT_REF = "main"
EXPECTED_COMMIT = "87c233d"
NEW_RUN = False  # False: resume the last run of this cell if its settings did not change
RETRY_FAILED = True  # on resume, run crashed or failed cases again
STAGE_TO_LOCAL = True  # copy models from the Drive cache to the local disk before loading
# ==== END OF SETTINGS ================================================================
# ======================================================================================
# memopro Colab T4 heavy validation: shared bootstrap (embedded in every cell by build.py)
# Drive layout (DRIVE_ROOT, default /content/drive/MyDrive/memopro_colab):
#   install/        put memopro-src.tar.gz (source bundle: memopro + memopro-preload, 0197),
#                   or memopro-*.whl / memopro-*.tar.gz (sdist) here
#   hf_cache/       Hugging Face cache (models and datasets persist across sessions)
#   results/<cell>/<run_id>/   env.json, config.json, cases/*.json, logs/*.log,
#                              timelines/*.csv, summary.md, summary.csv, progress.jsonl
# ======================================================================================
import csv
import datetime
import glob
import hashlib
import json
import os
import shutil
import subprocess
import sys
import threading
import time

IN_COLAB = "google.colab" in sys.modules or os.path.isdir("/content")
LOCAL_SMOKE = os.environ.get("MP_LOCAL_SMOKE") == "1"  # local check of the cell logic, no Colab


LOG_FILE = None  # set by bootstrap: the cell's own progress log inside the run folder (0096)


def _log(msg):
    line = f"[{datetime.datetime.now():%H:%M:%S}] {msg}"
    print(line, flush=True)
    if LOG_FILE:
        with open(LOG_FILE, "a") as f:
            f.write(line + "\n")


def mount_drive():
    if LOCAL_SMOKE:
        root = os.environ["MP_DRIVE_ROOT"]
    else:
        from google.colab import drive  # noqa: PLC0415

        if not os.path.isdir("/content/drive/MyDrive"):
            drive.mount("/content/drive")
        root = DRIVE_ROOT
    for sub in ("install", "hf_cache", "results"):
        os.makedirs(os.path.join(root, sub), exist_ok=True)
    return root


def _pip(*args):
    return subprocess.run([sys.executable, "-m", "pip", "install", "-q", *args],
                          capture_output=True, text=True, check=False)


def _rust():
    cargo = os.path.expanduser("~/.cargo/bin")
    if not os.path.exists(os.path.join(cargo, "cargo")):
        subprocess.run("curl -sSf https://sh.rustup.rs | sh -s -- -y --profile minimal",
                       shell=True, check=True, capture_output=True)
    os.environ["PATH"] = cargo + os.pathsep + os.environ["PATH"]


def _build_source(src, cache):
    """memopro from a source tree (Rust core via maturin) and the Linux memopro-preload library.
    The built wheel and library are kept in ``cache`` (Drive, per bundle) so later sessions skip
    the 5-8 minute build (0224)."""
    lib_name = "libmemopro_preload.so"
    wheels = sorted(glob.glob(os.path.join(cache, "memopro-*.whl")))
    if wheels and os.path.exists(os.path.join(cache, lib_name)):
        r = _pip("--force-reinstall", "--no-deps", wheels[-1])
        lib = os.path.join("/content", lib_name)
        shutil.copyfile(os.path.join(cache, lib_name), lib)
        os.environ["MP_PRELOAD_LIB"] = lib
        return {"ok": r.returncode == 0, "stderr": r.stderr[-1500:] if r.returncode else "",
                "preload": lib, "cached_build": cache}
    _rust()
    _pip("maturin>=1.9,<2.0")
    os.makedirs(cache, exist_ok=True)
    w = subprocess.run([sys.executable, "-m", "pip", "wheel", "-q", "--no-deps", src, "-w", cache],
                       capture_output=True, text=True, check=False)
    wheels = sorted(glob.glob(os.path.join(cache, "memopro-*.whl")))
    r = _pip("--force-reinstall", "--no-deps", wheels[-1]) if wheels else w
    info = {"ok": bool(wheels) and r.returncode == 0,
            "stderr": (w.stderr + r.stderr)[-1500:] if r.returncode or not wheels else ""}
    b = subprocess.run(["cargo", "build", "-q", "--release", "-p", "memopro-preload"], cwd=src,
                       capture_output=True, text=True, check=False)
    lib = os.path.join(src, "target", "release", lib_name)
    if b.returncode == 0 and os.path.exists(lib):
        os.environ["MP_PRELOAD_LIB"] = lib
        info["preload"] = lib
        shutil.copyfile(lib, os.path.join(cache, lib_name))
    else:
        info["preload_error"] = b.stderr[-1000:]
    return info


def install_memopro(root):
    """source bundle in install/ (memopro + memopro-preload) > wheel > sdist (builds the Rust
    core) > GitHub with a Colab secret."""
    if LOCAL_SMOKE:
        import memopro  # noqa: PLC0415

        return {"source": "local environment", "version": memopro.__version__}
    try:
        import memopro  # noqa: PLC0415

        if os.environ.get("MP_MEMOPRO_INSTALLED") == "1":
            return {"source": "already installed in this session", "version": memopro.__version__,
                    "commit": os.environ.get("MP_MEMOPRO_COMMIT")}
    except ImportError:
        pass
    if globals().get("SOURCE_REF"):  # a diagnosis run (0354): the public repository at this
        # branch, built here (Rust core and memopro-preload), its tests from the same checkout
        src = "/content/memopro-src"
        shutil.rmtree(src, ignore_errors=True)
        c = subprocess.run(["git", "clone", "-q", "--depth", "1", "--branch", SOURCE_REF,
                            "https://github.com/imhyensuk/memopro.git", src],
                           capture_output=True, text=True, check=False)
        if c.returncode != 0:
            raise RuntimeError(f"cloning memopro@{SOURCE_REF} failed: {c.stderr[-800:]}")
        sha = subprocess.run(["git", "rev-parse", "HEAD"], cwd=src, capture_output=True,
                             text=True, check=True).stdout.strip()[:12]
        _log(f"building memopro@{SOURCE_REF} ({sha}); ~5-8 min the first time per commit")
        info = {"source": f"github@{SOURCE_REF}", "commit": sha,
                **_build_source(src, os.path.join(root, "install", f"built-{sha}"))}
        if not info.get("ok"):
            raise RuntimeError(f"memopro build failed: {info}")
        os.environ["MP_MEMOPRO_INSTALLED"] = "1"
        os.environ["MP_MEMOPRO_COMMIT"] = sha
        return info
    if globals().get("PIP_SPEC"):  # a published release (E068): PyPI, and the repository at
        # GIT_REF for its tests (public, no token)
        _log(f"installing {PIP_SPEC} from PyPI")
        r = _pip("--pre", PIP_SPEC)
        src = "/content/memopro-src"
        shutil.rmtree(src, ignore_errors=True)
        c = subprocess.run(["git", "clone", "-q", "--depth", "1", "--branch", GIT_REF,
                            "https://github.com/imhyensuk/memopro.git", src],
                           capture_output=True, text=True, check=False)
        info = {"source": "pypi", "spec": PIP_SPEC, "ok": r.returncode == 0,
                "stderr": r.stderr[-1500:] if r.returncode else "",
                "tests_checkout": GIT_REF if c.returncode == 0 else c.stderr[-500:]}
        if not info["ok"]:
            raise RuntimeError(f"memopro install failed: {info}")
        os.environ["MP_MEMOPRO_INSTALLED"] = "1"
        return info
    info = {}
    bundles = sorted(glob.glob(os.path.join(root, "install", "memopro-src*.tar.gz")))
    wheels = sorted(glob.glob(os.path.join(root, "install", "memopro-*linux*.whl")))
    sdists = sorted(glob.glob(os.path.join(root, "install", "memopro-*.tar.gz")))
    if bundles:
        _log("building memopro and memopro-preload from the source bundle (~5-8 min the first time)")
        src = "/content/memopro-src"
        shutil.rmtree(src, ignore_errors=True)
        os.makedirs(src)
        subprocess.run(["tar", "-xzf", bundles[-1], "-C", src], check=True)
        with open(bundles[-1], "rb") as f:
            key = hashlib.sha256(f.read()).hexdigest()[:12]
        info = {"source": "source bundle", "file": os.path.basename(bundles[-1]),
                **_build_source(src, os.path.join(root, "install", f"built-{key}"))}
    elif wheels:
        r = _pip("--force-reinstall", "--no-deps", wheels[-1])
        info = {"source": "wheel", "file": os.path.basename(wheels[-1]), "ok": r.returncode == 0}
    elif sdists:
        _log("building memopro from the sdist (installs a Rust toolchain once, ~3-5 min)")
        _rust()
        _pip("maturin>=1.9,<2.0")
        r = _pip("--force-reinstall", "--no-deps", sdists[-1])
        info = {"source": "sdist", "file": os.path.basename(sdists[-1]), "ok": r.returncode == 0,
                "stderr": r.stderr[-1500:] if r.returncode else ""}
    else:
        token = None
        try:
            from google.colab import userdata  # noqa: PLC0415

            token = userdata.get("GITHUB_TOKEN")
        except Exception:  # noqa: BLE001
            token = None
        if not token:
            raise RuntimeError(
                "no memopro package: put the sdist (memopro-*.tar.gz) or a Linux wheel in "
                f"{os.path.join(root, 'install')}, or add a GITHUB_TOKEN Colab secret"
            )
        src = "/content/memopro-src"
        shutil.rmtree(src, ignore_errors=True)
        c = subprocess.run(["git", "clone", "-q", "--depth", "1", "--branch", GIT_REF,
                            f"https://{token}@github.com/imhyensuk/memopro.git", src],
                           capture_output=True, text=True, check=False)
        info = {"source": f"github@{GIT_REF}", "ok": False,
                "stderr": c.stderr[-800:].replace(token, "***")}
        if c.returncode == 0:
            info.update(_build_source(src))
            info["stderr"] = info.get("stderr", "").replace(token, "***")
    if not info.get("ok"):
        raise RuntimeError(f"memopro install failed: {info}")
    os.environ["MP_MEMOPRO_INSTALLED"] = "1"
    return info


def install_extras():
    if LOCAL_SMOKE:
        return {}
    want = ["bitsandbytes", "accelerate", "peft", "datasets"]
    missing = []
    for name in want:
        try:
            __import__(name)
        except ImportError:
            missing.append(name)
    if missing:
        _pip(*missing)
    removed = []
    try:  # Colab ships torchao 0.10, which peft >= 0.21 refuses when it adds any LoRA layer (0200)
        from importlib.metadata import version  # noqa: PLC0415

        major, minor = (int(x) for x in version("torchao").split(".")[:2])
        if (major, minor) < (0, 16):
            subprocess.run([sys.executable, "-m", "pip", "uninstall", "-y", "-q", "torchao"],
                           capture_output=True, check=False)
            removed.append(f"torchao {major}.{minor}")
    except Exception:  # noqa: BLE001 - not installed, or an unusual version string
        pass
    return {"installed": missing, "removed": removed}


def environment(install_info):
    import platform  # noqa: PLC0415

    env = {"time": datetime.datetime.now().isoformat(timespec="seconds"),
           "python": sys.version.split()[0], "platform": platform.platform(),
           "install": install_info, "expected_commit": EXPECTED_COMMIT}
    for mod in ("memopro", "torch", "transformers", "accelerate", "bitsandbytes", "peft",
                "datasets", "huggingface_hub"):
        try:
            env[mod] = __import__(mod).__version__
        except Exception as e:  # noqa: BLE001
            env[mod] = f"unavailable: {type(e).__name__}"
    try:
        import torch  # noqa: PLC0415

        env["cuda"] = torch.cuda.is_available()
        if env["cuda"]:
            p = torch.cuda.get_device_properties(0)
            env["gpu"] = {"name": p.name, "total_bytes": p.total_memory,
                          "capability": f"{p.major}.{p.minor}"}
    except Exception as e:  # noqa: BLE001
        env["torch_error"] = str(e)
    try:
        with open("/proc/meminfo") as f:
            env["host_total_bytes"] = int(f.readline().split()[1]) * 1024
    except OSError:
        pass
    env["local_disk_free_bytes"] = shutil.disk_usage("/content" if IN_COLAB and not LOCAL_SMOKE
                                                     else os.getcwd()).free
    try:  # memopro build contents: which decisions this build includes
        import memopro.rt.torch as RT  # noqa: PLC0415

        env["build_has"] = {"0195_cuda_stream": hasattr(RT, "_renamed")}
        env["preload_lib"] = os.environ.get("MP_PRELOAD_LIB")
    except Exception as e:  # noqa: BLE001
        env["build_has"] = f"unknown: {e}"
    return env


# ------------------------------------------------------------------ run directory, resume
class Run:
    def __init__(self, root, cell, config, new_run=False):
        base = os.path.join(root, "results", cell)
        os.makedirs(base, exist_ok=True)
        latest = os.path.join(base, "LATEST")
        rid = None
        wanted = globals().get("RESUME_RUN") or ""
        if wanted:  # a run chosen by id (0222): it must exist and have the same settings
            prev = os.path.join(base, wanted, "config.json")
            if not os.path.exists(prev) or json.load(open(prev)) != config:
                raise RuntimeError(f"RESUME_RUN {wanted!r}: no such run of {cell} with these settings")
            rid = wanted
            with open(latest, "w") as f:
                f.write(rid)
        elif not new_run and os.path.exists(latest):
            rid = open(latest).read().strip()
            prev = os.path.join(base, rid, "config.json")
            if not os.path.exists(prev) or json.load(open(prev)) != config:
                _log("config changed since the last run: starting a new run")
                rid = None
        if rid is None:
            rid = datetime.datetime.now().strftime("%Y%m%d-%H%M%S")
            with open(latest, "w") as f:
                f.write(rid)
        self.dir = os.path.join(base, rid)
        for sub in ("cases", "logs", "timelines", "reports"):
            os.makedirs(os.path.join(self.dir, sub), exist_ok=True)
        with open(os.path.join(self.dir, "config.json"), "w") as f:
            json.dump(config, f, indent=1)
        self.id, self.cell = rid, cell
        _log(f"results: {self.dir}")

    def path(self, *parts):
        return os.path.join(self.dir, *parts)

    def done(self, key):
        p = self.path("cases", key + ".json")
        if not os.path.exists(p):
            return None
        rec = json.load(open(p))
        if RETRY_FAILED and rec.get("status") in ("crashed", "error"):
            return None  # try a crashed or failed case again on resume (0200); a timeout is a
            # result (it re-ran a known 1-hour timeout in every session, 0220)
        return rec

    def save(self, key, rec):
        with open(self.path("cases", key + ".json"), "w") as f:
            json.dump(rec, f, indent=1, default=str)
        with open(self.path("progress.jsonl"), "a") as f:
            f.write(json.dumps({"t": time.time(), "case": key, "status": rec.get("status"),
                                "seconds": rec.get("wall_s")}) + "\n")

    def records(self):
        out = []
        for p in sorted(glob.glob(self.path("cases", "*.json"))):
            out.append(json.load(open(p)))
        return out


# ------------------------------------------------------------------ system timeline sampler
class Timeline:
    """nvidia-smi and host memory every `period` seconds while a case runs (CSV per case)."""

    def __init__(self, path, period=0.5):
        self.path, self.period, self.stop = path, period, threading.Event()
        self.peak_gpu_used, self.peak_host_used = 0, 0
        self.thread = threading.Thread(target=self._run, daemon=True)
        self.thread.start()

    def _gpu(self):
        try:
            out = subprocess.run(["nvidia-smi", "--query-gpu=memory.used,utilization.gpu",
                                  "--format=csv,noheader,nounits"], capture_output=True,
                                 text=True, timeout=5, check=False).stdout.strip().split(",")
            return int(out[0]) * 2**20, int(out[1])
        except Exception:  # noqa: BLE001
            return None, None

    def _host(self):
        try:
            info = {}
            with open("/proc/meminfo") as f:
                for line in f:
                    k, v = line.split(":", 1)
                    info[k] = int(v.split()[0]) * 1024
            return info["MemTotal"] - info["MemAvailable"], info.get("SwapTotal", 0) - info.get("SwapFree", 0)
        except Exception:  # noqa: BLE001
            return None, None

    def _run(self):
        t0 = time.time()
        with open(self.path, "w", newline="") as f:
            w = csv.writer(f)
            w.writerow(["t_s", "gpu_used_bytes", "gpu_util_pct", "host_used_bytes", "swap_used_bytes"])
            while not self.stop.is_set():
                g, u = self._gpu()
                h, s = self._host()
                w.writerow([round(time.time() - t0, 2), g, u, h, s])
                f.flush()
                if g:
                    self.peak_gpu_used = max(self.peak_gpu_used, g)
                if h:
                    self.peak_host_used = max(self.peak_host_used, h)
                self.stop.wait(self.period)

    def close(self):
        self.stop.set()
        self.thread.join(timeout=5)
        return {"timeline_peak_gpu_used_bytes": self.peak_gpu_used or None,
                "timeline_peak_host_used_bytes": self.peak_host_used or None}


# ------------------------------------------------------------------ workers
WORK_DIR = "/content/memopro_colab" if IN_COLAB and not LOCAL_SMOKE else os.path.join(
    os.environ.get("MP_DRIVE_ROOT", "."), "work")


def write_workers():
    os.makedirs(WORK_DIR, exist_ok=True)
    for name, source in WORKERS.items():
        with open(os.path.join(WORK_DIR, name), "w") as f:
            f.write(source)


def run_case(run, key, worker, args, timeout, env_extra=None, python=None):
    """One case in a fresh process; stdout/stderr to logs/<key>.log; result JSON to cases/."""
    prev = run.done(key)
    if prev is not None:
        _log(f"skip {key}: already {prev.get('status')} (resume)")
        return prev
    out_json = run.path("cases", key + ".tmp.json")
    log_path = run.path("logs", key + ".log")
    env = dict(os.environ)
    env.update({"HF_HOME": HF_HOME, "PYTHONUNBUFFERED": "1", "MP_OUT": out_json,
                "TOKENIZERS_PARALLELISM": "false"})
    env.update(env_extra or {})
    cmd = [python or sys.executable, os.path.join(WORK_DIR, worker), *[str(a) for a in args]]
    _log(f"run {key}")
    tl = Timeline(run.path("timelines", key + ".csv"))
    t0 = time.time()
    status = None
    with open(log_path, "w") as log:
        log.write("$ " + " ".join(cmd) + "\n")
        log.flush()
        # own process group: on timeout the whole group is killed, and we wait for it only a
        # minute (subprocess.run waits forever for a process that does not die, 0225)
        proc = subprocess.Popen(cmd, stdout=log, stderr=subprocess.STDOUT, env=env,
                                start_new_session=True)
        try:
            code = proc.wait(timeout=timeout)
            status = None if code == 0 else f"crashed (exit {code})"
        except subprocess.TimeoutExpired:
            status = "timeout"
            try:
                os.killpg(proc.pid, 9)
            except ProcessLookupError:
                pass
            try:
                proc.wait(timeout=60)
            except subprocess.TimeoutExpired:
                status = "timeout (the process did not exit after SIGKILL)"
    extra = tl.close()
    rec = {}
    if os.path.exists(out_json):
        rec = json.load(open(out_json))
        os.remove(out_json)
    if status and not rec.get("status"):
        rec["status"] = "timeout" if status.startswith("timeout") else "crashed"
        rec["exit"] = status
        with open(log_path) as f:
            rec["log_tail"] = f.read()[-3000:]
    rec.setdefault("status", "ok")
    rec.update({"case": key, "wall_s": round(time.time() - t0, 1), **extra})
    run.save(key, rec)
    _log(f"  -> {rec['status']} ({rec['wall_s']}s)")
    return rec


# ------------------------------------------------------------------ models on Drive / local disk
PATTERNS = ["*.json", "*.safetensors", "*.model", "*.txt", "tokenizer*", "*.tiktoken"]


def fetch_model(repo, drive=True):
    """Download once into the Drive cache; copy to local disk for fast loading when it fits.
    ``drive=False``: straight to the local disk only (models too big for the Drive, 0240)."""
    if LOCAL_SMOKE:
        return repo
    from huggingface_hub import snapshot_download  # noqa: PLC0415

    if not drive:
        local = os.path.join("/content/models", local_name(repo))
        t = time.time()
        snapshot_download(repo, local_dir=local, allow_patterns=PATTERNS,
                          cache_dir="/content/hf_fresh")
        _log(f"downloaded {repo} to local disk only ({time.time() - t:.0f}s)")
        return local

    t = time.time()
    path = snapshot_download(repo, cache_dir=os.path.join(HF_HOME, "hub"), allow_patterns=PATTERNS)
    _log(f"model {repo} on Drive ({time.time() - t:.0f}s)")
    if not STAGE_TO_LOCAL:
        return path
    files = {os.path.basename(p): _blob(p) for p in glob.glob(os.path.join(path, "*"))}
    size = sum(os.path.getsize(p) for p in files.values())
    local = os.path.join("/content/models", local_name(repo))
    broken = [n for n, p in files.items() if _link_text(p)]
    if broken:  # the Drive cache lost the files behind its links: download straight to local
        _log(f"{repo}: Drive cache entries {broken} are link text without their blob; "
             "downloading to the local disk instead")
        shutil.rmtree(local, ignore_errors=True)
        t = time.time()
        # a fresh download, with a cache on the local disk: given local_dir alone, the hub reused
        # the broken Drive cache and "downloaded" the 79-byte file again in 1 s (0226)
        snapshot_download(repo, local_dir=local, allow_patterns=PATTERNS, force_download=True,
                          cache_dir="/content/hf_fresh")
        still = [n for n in os.listdir(local)
                 if os.path.isfile(os.path.join(local, n)) and _link_text(os.path.join(local, n))]
        if still:
            raise RuntimeError(f"{repo}: {still} are still link text after a fresh download")
        size = sum(os.path.getsize(os.path.join(local, n)) for n in os.listdir(local)
                   if os.path.isfile(os.path.join(local, n)))
        _log(f"downloaded {repo} to local disk ({size / 2**30:.1f} GiB, {time.time() - t:.0f}s)")
        return local
    if os.path.isdir(local) and all(
            os.path.exists(os.path.join(local, n))
            and os.path.getsize(os.path.join(local, n)) == os.path.getsize(p)
            for n, p in files.items()):
        return local
    free = shutil.disk_usage("/content").free
    if free < size + 10 * 2**30:
        _log(f"not enough local disk to stage {repo} ({size / 2**30:.1f} GiB + 10 GiB margin, "
             f"{free / 2**30:.1f} GiB free); loading from Drive")
        return path
    t = time.time()
    partial = local + ".partial"  # a copy cut short (disconnect) must not look complete (0100)
    shutil.rmtree(partial, ignore_errors=True)
    shutil.rmtree(local, ignore_errors=True)
    os.makedirs(partial)
    for name, src in files.items():  # the blobs themselves, not what Drive made of the links
        shutil.copyfile(src, os.path.join(partial, name))
    os.rename(partial, local)
    _log(f"staged {repo} to local disk ({size / 2**30:.1f} GiB, {time.time() - t:.0f}s)")
    return local


def _link_text(p):
    """The link target written in a small file, if that is what ``p`` is ("../../blobs/...")."""
    if os.path.getsize(p) >= 512:
        return None
    with open(p, "rb") as f:
        text = f.read().decode("utf-8", "replace").strip()
    return text if text.startswith("../../blobs/") else None


def _blob(p):
    """The file a Hugging Face snapshot entry stands for. On Google Drive the cache's relative
    symlinks can turn into small text files holding the link ("../../blobs/..."): a 79-byte
    "model.safetensors" was staged instead of DINOv2-giant's 4.5 GB (0222). Follow such a text
    when its target exists; otherwise the text file itself is returned (and caught as broken)."""
    real = os.path.realpath(p)
    text = _link_text(real)
    if text:
        target = os.path.normpath(os.path.join(os.path.dirname(p), text))
        if os.path.exists(target):
            return target
    return real


def local_name(repo):
    return repo.replace("/", "--")


def free_local_models(keep=()):
    for d in glob.glob("/content/models/*"):
        if os.path.basename(d) not in keep:
            shutil.rmtree(d, ignore_errors=True)


# ------------------------------------------------------------------ summaries
def fmt_bytes(n):
    return "-" if not isinstance(n, (int, float)) else f"{n / 2**30:.2f} GiB"


def fmt(v, digits=2):
    if v is None:
        return "-"
    if isinstance(v, float):
        return f"{v:.{digits}f}"
    return str(v)


def write_summary(run, title, columns, rows, notes=()):
    """columns: list of (header, function(record) -> value)."""
    with open(run.path("summary.csv"), "w", newline="") as f:
        w = csv.writer(f)
        w.writerow([h for h, _ in columns])
        for r in rows:
            w.writerow([fn(r) for _, fn in columns])
    lines = [f"# {title}", "", f"run `{run.id}`, cell `{run.cell}`", ""]
    lines.append("| " + " | ".join(h for h, _ in columns) + " |")
    lines.append("|" + "---|" * len(columns))
    for r in rows:
        lines.append("| " + " | ".join(str(fn(r)) for _, fn in columns) + " |")
    lines += ["", *notes]
    text = "\n".join(lines) + "\n"
    with open(run.path("summary.md"), "w") as f:
        f.write(text)
    archive = shutil.make_archive(run.dir, "zip", run.dir)
    _log(f"summary: {run.path('summary.md')}\narchive to share: {archive}")
    print(text)
    return text


def bootstrap(cell, config):
    root = mount_drive()
    global HF_HOME
    HF_HOME = os.environ.get("MP_HF_HOME") or os.path.join(root, "hf_cache")
    os.environ["HF_HOME"] = HF_HOME
    info = install_memopro(root)
    extras = install_extras()
    write_workers()
    if info.get("commit"):  # a new commit is a new run
        config = {**config, "COMMIT": info["commit"]}
    run = Run(root, cell, config, new_run=NEW_RUN)
    global LOG_FILE
    LOG_FILE = run.path("orchestrator.log")
    env = environment({**info, "extras": extras})
    with open(run.path("env.json"), "w") as f:
        json.dump(env, f, indent=1, default=str)
    _log(json.dumps({k: env.get(k) for k in ("memopro", "torch", "transformers", "gpu",
                                                "build_has")}, default=str))
    return run, env


# ------------------------------------------------------------------ shared by the suite cells (0240)
MIB = 1 << 20
DETERMINISTIC = {"CUBLAS_WORKSPACE_CONFIG": ":4096:8"}
VENV = "/content/venv-unsloth"
T_START = time.time()  # the cell started (common is inlined at its top)


def left():
    """Seconds left in the cell's TIME_LIMIT_S (no limit if the cell sets none)."""
    return globals().get("TIME_LIMIT_S", 10**9) - (time.time() - T_START)


def case(run, key, expected_s, worker, args, timeout, env_extra=None, python=None):
    """run_case, unless the case would end after the time limit (then recorded as skipped)."""
    prev = run.done(key)
    if prev is not None and prev.get("status") != "skipped":  # finished: run_case skips it
        return run_case(run, key, worker, args, timeout, env_extra, python)
    if prev is not None:  # skipped for time last session: try again now
        os.remove(run.path("cases", key + ".json"))
    if expected_s > left():
        rec = {"case": key, "status": "skipped", "why": f"time budget: {left() / 60:.0f} min left, "
               f"case expects {expected_s / 60:.0f} min"}
        run.save(key, rec)
        _log(f"skip {key}: {rec['why']}")
        return rec
    # never past the limit by more than 5 minutes, even if the case hangs
    timeout = min(timeout, int(max(expected_s, left())) + 300)
    return run_case(run, key, worker, args, timeout, env_extra, python)


def short(model):
    return model.split("/")[-1]


def unsloth_python(run):
    py = os.path.join(VENV, "bin", "python")
    if not os.path.exists(py):
        # uv, not `python -m venv`: Colab's Python has no ensurepip, so venv failed (0238)
        _log("installing Unsloth into its own virtual environment with uv (~3-6 min)")
        steps = [[sys.executable, "-m", "pip", "install", "-q", "uv"],
                 [sys.executable, "-m", "uv", "venv", "-q", VENV],
                 [sys.executable, "-m", "uv", "pip", "install", "-q", "--python", py, "unsloth",
                  "datasets"]]
        for cmd in steps:
            r = subprocess.run(cmd, capture_output=True, text=True, check=False)
            if r.returncode:
                _log(f"Unsloth install failed at {cmd[2:5]}: {(r.stdout + r.stderr)[-1500:]}")
                break
    v = subprocess.run([py, "-c", "import unsloth, torch, transformers; print(unsloth.__version__, "
                        "torch.__version__, transformers.__version__)"],
                       capture_output=True, text=True, check=False)
    info = {"versions": v.stdout.strip(), "error": v.stderr[-1500:] if v.returncode else ""}
    with open(run.path("unsloth_env.json"), "w") as f:
        json.dump(info, f, indent=1)
    _log(f"Unsloth environment: {info['versions'] or info['error'][-300:]}")
    return py


WORKERS = {
    'worker_common.py': r'''"""Shared helpers for the Colab workers (each case runs in a fresh process)."""

import gc
import json
import os
import resource
import sys
import threading
import time
import traceback

import torch

DEVICE = os.environ.get("MP_DEVICE") or (
    "cuda" if torch.cuda.is_available()
    else "mps" if getattr(torch.backends, "mps", None) and torch.backends.mps.is_available()
    else "cpu"
)
RESULT: dict = {"status": None, "device": DEVICE, "argv": sys.argv[1:]}
_T0 = time.time()


def sync():
    if DEVICE == "cuda":
        torch.cuda.synchronize()
    elif DEVICE == "mps":
        torch.mps.synchronize()


def peak_rss():
    """This process's peak resident memory. On Linux from /proc/self/status (VmHWM): getrusage's
    ru_maxrss survives execve, so a worker started from a large notebook kernel reported the
    kernel's peak (5.19 GiB in every case of E042's first run, 0199)."""
    try:
        with open("/proc/self/status") as f:
            for line in f:
                if line.startswith("VmHWM:"):
                    return int(line.split()[1]) * 1024
    except OSError:
        pass
    rss = resource.getrusage(resource.RUSAGE_SELF).ru_maxrss
    return rss if sys.platform == "darwin" else rss * 1024


class DeviceMonitor:
    """Peak device memory in use (cudaMemGetInfo: includes the CUDA context and other processes'
    use, i.e. what nvidia-smi shows) sampled every 20 ms, plus the allocator's own peaks."""

    def __init__(self):
        self.peak_used = 0
        self._stop = threading.Event()
        if DEVICE == "cuda":
            threading.Thread(target=self._run, daemon=True).start()

    def _run(self):
        while not self._stop.is_set():
            free, total = torch.cuda.mem_get_info()
            self.peak_used = max(self.peak_used, total - free)
            self._stop.wait(0.02)

    def reset(self):
        self.peak_used = 0
        if DEVICE == "cuda":
            sync()
            torch.cuda.reset_peak_memory_stats()

    def snapshot(self):
        out = {"host_peak_rss_bytes": peak_rss()}
        if DEVICE == "cuda":
            sync()
            free, total = torch.cuda.mem_get_info()
            out.update(allocated_bytes=torch.cuda.memory_allocated(),
                       reserved_bytes=torch.cuda.memory_reserved(),
                       peak_allocated_bytes=torch.cuda.max_memory_allocated(),
                       peak_reserved_bytes=torch.cuda.max_memory_reserved(),
                       device_used_bytes=total - free,
                       peak_device_used_bytes=max(self.peak_used, total - free))
        elif DEVICE == "mps":
            out.update(allocated_bytes=torch.mps.current_allocated_memory(),
                       driver_bytes=torch.mps.driver_allocated_memory())
        return out


MON = DeviceMonitor()
if DEVICE == "cuda" and os.environ.get("MP_GPU_LIMIT"):  # a smaller GPU, emulated (E049, 0240)
    _limit = int(os.environ["MP_GPU_LIMIT"])
    torch.cuda.set_per_process_memory_fraction(
        min(1.0, _limit / torch.cuda.get_device_properties(0).total_memory))
    RESULT["gpu_limit"] = _limit


def is_oom(e):
    text = f"{type(e).__name__} {e}".lower()
    return isinstance(e, torch.OutOfMemoryError) or "out of memory" in text


def cleanup():
    gc.collect()
    if DEVICE == "cuda":
        torch.cuda.empty_cache()
    elif DEVICE == "mps":
        torch.mps.empty_cache()


def memopro_report():
    try:
        import memopro

        return [{"technique": e.technique, "action": e.action, "detail": e.detail}
                for e in memopro.report().entries]
    except Exception as e:  # noqa: BLE001
        return [{"error": str(e)}]


def finish(status=None, **fields):
    RESULT.update(fields)
    if status:
        RESULT["status"] = status
    RESULT.setdefault("status", "ok")
    if RESULT["status"] is None:
        RESULT["status"] = "ok"
    RESULT["worker_s"] = round(time.time() - _T0, 1)
    RESULT["memory_end"] = MON.snapshot()
    RESULT["memopro_report"] = memopro_report()
    path = os.environ.get("MP_OUT")
    text = json.dumps(RESULT, default=str)
    if path:
        with open(path, "w") as f:
            f.write(text)
    print("RESULT " + text[:2000], flush=True)


def guarded(main):
    """Run main(); record an OOM or any other error as the case's result instead of crashing."""
    try:
        main()
    except Exception as e:  # noqa: BLE001
        status = "oom" if is_oom(e) else "error"
        finish(status, error=f"{type(e).__name__}: {e}"[:1500], trace=traceback.format_exc()[-3000:])
        return
    if RESULT.get("status") is None:
        finish()


def wikitext_ids(tokenizer, split, n_tokens):
    """WikiText-2 raw tokens (the revision pinned in 0017), joined with blank lines."""
    from datasets import load_dataset

    ds = load_dataset("Salesforce/wikitext", "wikitext-2-raw-v1",
                      revision="b08601e04326c79dfdd32d625aee71d232d685c3", split=split)
    text = "\n\n".join(t for t in ds["text"][: 20000 if split == "train" else None])
    ids = tokenizer(text, return_tensors="pt", verbose=False).input_ids[0]
    if ids.numel() < n_tokens:
        reps = n_tokens // ids.numel() + 1
        ids = ids.repeat(reps)
    return ids[:n_tokens]


def model_path(model_id):
    return os.environ.get("MP_MODEL_PATH") or model_id
''',
    'worker_suite.py': r'''"""E042 worker (docs/research/0197): one case of the Colab T4 suite in a fresh process.

    worker_suite.py lm_gen    <model> <plain|memopro> <budget_bytes> <new_tokens>
    worker_suite.py lm_lora   <model> <plain|memopro> <budget_bytes> <steps> <seq>
    worker_suite.py vis_infer <model> <plain|memopro> <budget_bytes> <batch>
    worker_suite.py vis_lora  <model> <plain|memopro> <budget_bytes> <steps> <batch>
    worker_suite.py ft        <model> <ceiling_bytes> <steps> <seq>    (public API, E068)
    worker_suite.py gen       <model> <ceiling_bytes> <new_tokens>     (public API, E068)

`plain` loads the model normally onto the GPU; `memopro` streams it from host memory within
`budget_bytes` (memopro.rt.torch.stream_model(device="cuda"), 0195).
"""

import glob
import hashlib
import os
import sys
import time

import torch
from worker_common import DEVICE, MON, RESULT, cleanup, finish, guarded, model_path, sync

PROMPTS = [
    "Explain how a hash table handles collisions.",
    "Write a Python function that checks whether a string is a palindrome, with a short docstring.",
]
VISION_CLASSES = {"resnet": "ResNetForImageClassification", "dinov2": "Dinov2Model",
                  "vit": "ViTModel"}


def copy_stats(m):
    """Device copies, cache hits, prefetched weights and the cache size (0201), if streamed."""
    w = getattr(m, "memopro_weights", None)
    if w is None or not hasattr(w, "copy_stats"):
        return None
    return {**w.copy_stats, "gpu_budget": w.gpu_budget, "gpu_cache_bytes": w.gpu_cache_bytes}


def sha(t):
    return hashlib.sha256(t.detach().float().cpu().numpy().tobytes()).hexdigest()[:16]


def vision_class(model_id):
    import transformers

    for key, name in VISION_CLASSES.items():
        if key in model_id.lower():
            return getattr(transformers, name)
    return transformers.AutoModel


def model_files(path):
    """Name, size and first 16 bytes of each file in a model folder."""
    out = []
    for f in sorted(glob.glob(os.path.join(path, "*"))):
        with open(f, "rb") as fh:
            head = fh.read(16)
        out.append({"name": os.path.basename(f), "real": os.path.realpath(f),
                    "size": os.path.getsize(f), "head": head.hex()})
    return out


def load(kind, model_id, mode, budget):
    import transformers

    import memopro.rt.torch as rtt

    path = model_path(model_id)
    cls = transformers.AutoModelForCausalLM if kind == "lm" else vision_class(model_id)
    t = time.time()
    if mode == "plain":
        dtype = torch.bfloat16 if kind == "lm" else None
        m = cls.from_pretrained(path, dtype=dtype).to(DEVICE).eval()
    else:
        try:
            gpu = os.environ.get("MP_GPU_BUDGET")  # with an emulated smaller GPU (E049)
            m = rtt.stream_model(path, budget=budget, device=DEVICE, model_class=cls,
                                 **({"gpu_budget": int(gpu)} if gpu else {}))
        except Exception:
            RESULT["files"] = model_files(path)  # what the streamer saw (DINOv2, 0220)
            raise
    sync()
    RESULT["load_s"] = round(time.time() - t, 1)
    RESULT["memory_after_load"] = MON.snapshot()
    return m, path


def lora_layers(m, device):
    """PEFT adapters of a streamed model start on the meta device: put them on the GPU (as
    memopro.finetune does)."""
    import peft

    for layer in m.modules():
        if isinstance(layer, peft.tuners.lora.LoraLayer):
            for name in layer.active_adapters:
                if layer.lora_A[name].weight.device.type == "meta":
                    layer.lora_A[name].to_empty(device=device)
                    layer.lora_B[name].to_empty(device=device)
                    layer.reset_lora_parameters(name, True)


def lm_gen(model_id, mode, budget, new_tokens):
    import transformers

    import memopro

    m, path = load("lm", model_id, mode, budget)
    tok = transformers.AutoTokenizer.from_pretrained(path)
    texts, secs = [], []
    for p in PROMPTS:
        t = time.time()
        if mode == "plain":
            msgs = [{"role": "user", "content": p}]
            ids = tok.apply_chat_template(msgs, add_generation_prompt=True, return_tensors="pt")
            ids = (ids["input_ids"] if hasattr(ids, "keys") else ids).to(DEVICE)
            with torch.no_grad():
                out = m.generate(input_ids=ids, max_new_tokens=new_tokens, do_sample=False,
                                 pad_token_id=tok.eos_token_id)
            texts.append(tok.decode(out[0, ids.shape[1]:], skip_special_tokens=True))
        else:
            texts.append(memopro.generate(m, p, tokenizer=tok, max_new_tokens=new_tokens))
        sync()
        secs.append(time.time() - t)
    finish(texts=texts, seconds=secs, s_per_token=sum(secs) / (new_tokens * len(PROMPTS)),
           rt=getattr(getattr(m, "memopro_runtime", None), "stats", lambda: None)(),
           copy_stats=copy_stats(m))


def lm_lora(model_id, mode, budget, steps, seq):
    import transformers

    import memopro
    import memopro.rt.torch as rtt
    from worker_common import wikitext_ids

    if mode == "memopro":
        m, path = load("lm", model_id, mode, budget)
        tok = transformers.AutoTokenizer.from_pretrained(path)
        ids = wikitext_ids(tok, "train", (seq - 1) * steps)
        texts = [tok.decode(ids[i * (seq - 1):(i + 1) * (seq - 1)]) for i in range(steps)]
        r = memopro.finetune(m, texts, tokenizer=tok, seq_len=seq)
        finish(losses=r.losses, loss_bits=[float.hex(x) for x in r.losses],
               step_s=r.step_seconds, tokens=r.tokens, rt=m.memopro_runtime.stats())
        return
    import peft

    m, path = load("lm", model_id, mode, budget)
    tok = transformers.AutoTokenizer.from_pretrained(path)
    ids = wikitext_ids(tok, "train", (seq - 1) * steps)
    texts = [tok.decode(ids[i * (seq - 1):(i + 1) * (seq - 1)]) for i in range(steps)]
    eos = tok.eos_token or ""
    all_ids = tok(eos.join(texts) + eos, return_tensors="pt").input_ids[0]
    torch.manual_seed(0)
    m = peft.get_peft_model(m, peft.LoraConfig(r=8, lora_alpha=16,
                                               target_modules=["q_proj", "k_proj", "v_proj", "o_proj"]))
    for p in m.parameters():
        if p.requires_grad:
            p.data = p.data.float()
    m.train()
    m.base_model.model.gradient_checkpointing_enable(gradient_checkpointing_kwargs={"use_reentrant": True})
    m.base_model.model.enable_input_require_grads()
    opt = torch.optim.AdamW([p for p in m.parameters() if p.requires_grad], lr=2e-4)
    losses, step_s = [], []
    pieces = [all_ids[i:i + seq] for i in range(0, len(all_ids) - 1, seq)]
    for piece in [p for p in pieces if len(p) > 1]:
        t = time.time()
        loss = rtt.causal_lm_loss(m.base_model.model, piece.unsqueeze(0).to(DEVICE))
        loss.backward()
        opt.step()
        opt.zero_grad(set_to_none=True)
        sync()
        losses.append(float(loss.detach()))
        step_s.append(time.time() - t)
    finish(losses=losses, loss_bits=[float.hex(x) for x in losses], step_s=step_s)


def lora_texts(tok, steps, seq):
    from worker_common import wikitext_ids

    ids = wikitext_ids(tok, "train", (seq - 1) * steps)
    return [tok.decode(ids[i * (seq - 1):(i + 1) * (seq - 1)]) for i in range(steps)]


def ft(model_id, ceiling, steps, seq):
    """memopro.finetune as a user calls it: a folder and a ceiling for the whole process (0344)."""
    import transformers

    import memopro

    path = model_path(model_id)
    tok = transformers.AutoTokenizer.from_pretrained(path)
    r = memopro.finetune(path, lora_texts(tok, steps, seq), tokenizer=tok, budget=ceiling,
                         seq_len=seq)
    finish(losses=r.losses, loss_bits=[float.hex(x) for x in r.losses], step_s=r.step_seconds,
           tokens=r.tokens, held_back=r.held_back, ceiling=ceiling,
           rt=r.model.memopro_runtime.stats(), copy_stats=copy_stats(r.model))


def gen(model_id, ceiling, new_tokens):
    """memopro.generate as a user calls it, one call per prompt, the same ceiling (0344)."""
    import transformers

    import memopro

    path = model_path(model_id)
    tok = transformers.AutoTokenizer.from_pretrained(path)
    texts, secs = [], []
    for p in PROMPTS:  # each call from the folder, as a user would call it
        t = time.time()
        texts.append(memopro.generate(path, p, tokenizer=tok, budget=ceiling,
                                      max_new_tokens=new_tokens))
        sync()
        secs.append(time.time() - t)
    finish(texts=texts, seconds=secs, s_per_token=sum(secs) / (new_tokens * len(PROMPTS)),
           ceiling=ceiling)


def images(batch, size):
    g = torch.Generator().manual_seed(0)
    return torch.rand(batch, 3, size, size, generator=g)


def features(m, x):
    out = m(pixel_values=x)
    logits = getattr(out, "logits", None)
    return logits if logits is not None else out.last_hidden_state[:, 0]


def vis_infer(model_id, mode, budget, batch):
    m, _ = load("vision", model_id, mode, budget)
    x = images(batch, 224).to(DEVICE)
    with torch.no_grad():
        t = time.time()
        y = features(m, x)
        sync()
        first = time.time() - t
        t = time.time()
        y2 = features(m, x)
        sync()
    finish(output_sha=sha(y), same_twice=bool(torch.equal(y, y2)), shape=list(y.shape),
           first_s=first, second_s=time.time() - t,
           rt=getattr(getattr(m, "memopro_runtime", None), "stats", lambda: None)(),
           copy_stats=copy_stats(m))


def vis_lora(model_id, mode, budget, steps, batch):
    """LoRA on the attention projections + a fresh linear head (10 classes) over the class token;
    the same random images and labels at every step."""
    import peft

    import memopro.rt.torch as rtt

    m, _ = load("vision", model_id, mode, budget)
    names = {n.split(".")[-1] for n, mod in m.named_modules() if isinstance(mod, torch.nn.Linear)}
    targets = [t for t in ("query", "value", "q_proj", "v_proj") if t in names]
    torch.manual_seed(0)
    m = peft.get_peft_model(m, peft.LoraConfig(r=8, lora_alpha=16, target_modules=targets))
    lora_layers(m, DEVICE)
    for p in m.parameters():
        if p.requires_grad:
            p.data = p.data.float()
    torch.manual_seed(1)
    hidden = m.base_model.model.config.hidden_size
    head = torch.nn.Linear(hidden, 10).to(DEVICE)
    params = [p for p in m.parameters() if p.requires_grad] + list(head.parameters())
    opt = torch.optim.AdamW(params, lr=1e-3)
    x = images(batch, 224).to(DEVICE)
    y = torch.arange(batch, device=DEVICE) % 10
    streamed = hasattr(m.base_model.model, "memopro_weights")
    losses, step_s = [], []
    for _ in range(steps):
        t = time.time()
        ctx = rtt.saved_weights(m.base_model.model) if streamed else torch.enable_grad()
        with ctx:
            f = features(m.base_model.model, x).float()
            loss = torch.nn.functional.cross_entropy(head(f), y)
            loss.backward()
        opt.step()
        opt.zero_grad(set_to_none=True)
        sync()
        losses.append(float(loss.detach()))
        step_s.append(time.time() - t)
    finish(losses=losses, loss_bits=[float.hex(v) for v in losses], step_s=step_s,
           targets=targets,
           rt=m.base_model.model.memopro_runtime.stats() if streamed else None)


def main():
    if sys.argv[1] in ("ft", "gen"):  # the public API: <model> <ceiling> <...> (E068)
        kind, model_id, ceiling, *rest = sys.argv[1:]
        RESULT.update(kind=kind, model=model_id, mode="memopro", budget=int(ceiling))
        torch.manual_seed(0)
        if kind == "ft":
            torch.use_deterministic_algorithms(True, warn_only=True)
        (ft if kind == "ft" else gen)(model_id, int(ceiling), *map(int, rest))
        cleanup()
        return
    kind, model_id, mode, budget, *rest = sys.argv[1:]
    RESULT.update(kind=kind, model=model_id, mode=mode, budget=int(budget))
    torch.manual_seed(0)
    if kind.endswith("_lora"):  # the same steps give the same bits (CUBLAS_WORKSPACE_CONFIG set)
        torch.use_deterministic_algorithms(True, warn_only=True)
    fn = {"lm_gen": lm_gen, "lm_lora": lm_lora, "vis_infer": vis_infer, "vis_lora": vis_lora}[kind]
    fn(model_id, mode, int(budget), *map(int, rest))
    cleanup()


if __name__ == "__main__":
    guarded(main)
''',
    'worker_unsloth.py': r'''"""E046 (docs/research/0227): the same 16-bit LoRA with Unsloth, run in its own virtual environment.

    python worker_unsloth.py <model> <bits 16|4> <steps> <seq>
"""

import sys
import time

import unsloth  # noqa: F401  (before transformers/peft, as Unsloth asks)
import torch
from unsloth import FastLanguageModel
from worker_common import DEVICE, MON, RESULT, finish, guarded, model_path, sync, wikitext_ids


def main():
    model_id, bits, steps, seq = sys.argv[1], int(sys.argv[2]), int(sys.argv[3]), int(sys.argv[4])
    RESULT.update(kind="unsloth_lora", model=model_id, bits=bits,
                  unsloth=getattr(unsloth, "__version__", "?"), torch=torch.__version__)
    t = time.time()
    m, tok = FastLanguageModel.from_pretrained(model_name=model_path(model_id), max_seq_length=seq,
                                               dtype=None, load_in_4bit=bits == 4)
    RESULT["load_s"] = round(time.time() - t, 1)
    RESULT["dtype"] = str(m.dtype)
    RESULT["memory_after_load"] = MON.snapshot()
    m = FastLanguageModel.get_peft_model(m, r=8, lora_alpha=16, lora_dropout=0, bias="none",
                                         target_modules=["q_proj", "k_proj", "v_proj", "o_proj"],
                                         use_gradient_checkpointing="unsloth", random_state=0)
    ids = wikitext_ids(tok, "train", (seq - 1) * steps)
    texts = [tok.decode(ids[i * (seq - 1):(i + 1) * (seq - 1)]) for i in range(steps)]
    eos = tok.eos_token or ""
    all_ids = tok(eos.join(texts) + eos, return_tensors="pt").input_ids[0]
    pieces = [all_ids[i:i + seq] for i in range(0, len(all_ids) - 1, seq)]
    opt = torch.optim.AdamW([p for p in m.parameters() if p.requires_grad], lr=2e-4)
    m.train()
    losses, step_s = [], []
    for piece in [p for p in pieces if len(p) > 1]:
        x = piece.unsqueeze(0).to(DEVICE)
        t = time.time()
        loss = m(input_ids=x, labels=x).loss
        loss.backward()
        opt.step()
        opt.zero_grad(set_to_none=True)
        sync()
        losses.append(float(loss.detach()))
        step_s.append(time.time() - t)
    finish(losses=losses, loss_bits=[float.hex(x) for x in losses], step_s=step_s,
           tokens=int(sum(len(p) for p in pieces if len(p) > 1)))


if __name__ == "__main__":
    guarded(main)
''',
}

# ---------------------------------------------------------------- body
def unsloth_body():
    config = {"MODELS": MODELS, "LORA": LORA, "UNSLOTH_4BIT": UNSLOTH_4BIT}
    run, env = bootstrap("unsloth", config)
    py = unsloth_python(run)
    steps, seq = LORA["steps"], LORA["seq"]
    for model, budgets in MODELS.items():
        free_local_models(keep=(local_name(model),))
        path = fetch_model(model)
        extra = {"MP_MODEL_PATH": path}
        n = short(model)
        run_case(run, f"{n}__P", "worker_suite.py", ["lm_lora", model, "plain", 0, steps, seq],
                 TIMEOUT_S, {**extra, **DETERMINISTIC})
        for b in budgets:
            run_case(run, f"{n}__M_{b // MIB}", "worker_suite.py",
                     ["lm_lora", model, "memopro", b, steps, seq], TIMEOUT_S,
                     {**extra, **DETERMINISTIC})
        run_case(run, f"{n}__U16", "worker_unsloth.py", [model, 16, steps, seq], TIMEOUT_S,
                 extra, python=py)
        if model in UNSLOTH_4BIT:
            run_case(run, f"{n}__U4", "worker_unsloth.py", [model, 4, steps, seq], TIMEOUT_S,
                     extra, python=py)
    free_local_models()
    summarize_unsloth(run)


def step_s(r):
    s = r.get("step_s") or []
    return sum(s[1:]) / len(s[1:]) if len(s) > 1 else None


def gpu_peak(r):
    return r.get("timeline_peak_gpu_used_bytes") or (r.get("memory_end") or {}).get(
        "peak_device_used_bytes")


def host_peak(r):
    return (r.get("memory_end") or {}).get("host_peak_rss_bytes")


def max_diff(a, b):
    a, b = a.get("losses") or [], b.get("losses") or []
    return max((abs(x - y) for x, y in zip(a, b)), default=None) if a and b else None


def summarize_unsloth(run):
    recs = {r.get("case"): r for r in run.records()}
    rows = []

    def check(name, criterion, result, detail):
        rows.append({"check": name, "criterion": criterion,
                     "result": "pass" if result is True else ("fail" if result is False else "report"),
                     "detail": detail})

    def ok(r):
        return r.get("status") == "ok" and len(r.get("losses") or []) == LORA["steps"]

    m3, m7 = short(list(MODELS)[0]), short(list(MODELS)[1])
    u16 = recs.get(f"{m7}__U16", {})
    ms7 = [recs.get(f"{m7}__M_{b // MIB}", {}) for b in MODELS[list(MODELS)[1]]]
    check("U1", "7B 16-bit: Unsloth does not complete; memopro completes at both budgets",
          u16.get("status") not in (None, "ok") and all(ok(m) for m in ms7),
          f"Unsloth {u16.get('status')} {str(u16.get('error', ''))[:160]}; memopro "
          f"{[m.get('status') for m in ms7]}")
    same = []
    for model, budgets in MODELS.items():
        ms = [recs.get(f"{short(model)}__M_{b // MIB}", {}) for b in budgets]
        same.append(all(ok(m) for m in ms) and ms[0].get("loss_bits") == ms[1].get("loss_bits"))
    check("U2", "memopro losses bit-identical across budgets (3B, 7B)", all(same), str(same))
    p3, m3r, u3 = (recs.get(f"{m3}__P", {}), recs.get(f"{m3}__M_{MODELS[list(MODELS)[0]][0] // MIB}", {}),
                   recs.get(f"{m3}__U16", {}))
    check("U3", "3B 16-bit: step time, GPU peak, host peak (Unsloth vs memopro vs PEFT)", None,
          f"step s P {fmt(step_s(p3))} / M {fmt(step_s(m3r))} / U16 {fmt(step_s(u3))}; GPU "
          f"{fmt_bytes(gpu_peak(p3))} / {fmt_bytes(gpu_peak(m3r))} / {fmt_bytes(gpu_peak(u3))}; "
          f"host {fmt_bytes(host_peak(p3))} / {fmt_bytes(host_peak(m3r))} / {fmt_bytes(host_peak(u3))}; "
          f"Unsloth dtype {u3.get('dtype')}")
    u4 = recs.get(f"{m7}__U4", {})
    first = lambda r: (r.get("losses") or [None])[0]  # noqa: E731
    check("U4", "7B: Unsloth 4-bit vs memopro 16-bit (step time, memory, first loss)", None,
          f"step s U4 {fmt(step_s(u4))} / M {fmt(step_s(ms7[0]))}; GPU {fmt_bytes(gpu_peak(u4))} / "
          f"{fmt_bytes(gpu_peak(ms7[0]))}; first loss {fmt(first(u4), 4)} / {fmt(first(ms7[0]), 4)}")
    check("U5", "3B loss difference from PEFT (max abs over steps)", None,
          f"memopro {fmt(max_diff(p3, m3r), 5)}, Unsloth {fmt(max_diff(p3, u3), 5)}")
    gate = rows[0]["result"] == "pass" and rows[1]["result"] == "pass"
    notes = [f"Gate (U1 and U2): **{'pass' if gate else 'fail'}**. Pre-registered: docs/research/0227.",
             "", "## All cases", "", "| case | status | load s | step s | GPU peak | host peak | first loss |",
             "|---|---|---|---|---|---|---|"]
    for key in sorted(recs):
        r = recs[key]
        notes.append(f"| {key} | {r.get('status')} | {fmt(r.get('load_s'), 1)} | {fmt(step_s(r))} | "
                     f"{fmt_bytes(gpu_peak(r))} | {fmt_bytes(host_peak(r))} | {fmt(first(r), 4)} |")
    errors = [(k, r) for k, r in sorted(recs.items()) if r.get("status") not in ("ok", None)]
    if errors:
        notes += ["", "## Errors", ""]
        for key, r in errors:
            text = (r.get("error") or r.get("exit") or "").replace("|", "/").replace("\n", " ")
            notes.append(f"- `{key}` ({r.get('status')}): {text[:400]}")
    cols = [("check", lambda r: r["check"]), ("criterion", lambda r: r["criterion"]),
            ("result", lambda r: r["result"]), ("detail", lambda r: r["detail"])]
    return write_summary(run, "memopro vs Unsloth on a T4 (E046)", cols, rows, notes)


unsloth_body()
